# bmesonML — Emulador XGBoost de la likelihood SMEFT19 (rotBIII)

Pipeline completo (estructura CosmoML):
- **Dataset**: muestreo LHS + SMEFT19 cacheado en disco; se genera sólo si no existe o `FORCE_REBUILD=True`
- **Preprocesamiento**: SMOTE sólo en train (sin data leakage) + KNN para likelihood sintética
- **Entrenamiento**: XGBoost con early stopping + curva de aprendizaje
- **SHAP**: beeswarm, bar, waterfall y dependence plots (estilo CosmoML)
- **Predicción**: función con penalización exponencial fuera del dominio físico

## Setup (run first)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time, csv, multiprocessing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from pathlib import Path
from scipy.stats.qmc import LatinHypercube, scale
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBRegressor, callback as xgb_cb

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size']   = 12

# ── Paths (notebook vive en notebooks/, repo root es ..) ──────────────────────
ROOT     = Path('..')
DATA_DIR = ROOT / 'data'
OUT_DIR  = ROOT / 'outputs'

DATASET_FILE = DATA_DIR / 'combined_rotBIII.dat'
BESTFIT_FILE = DATA_DIR / 'rotBIII.yaml'
MODEL_OUT    = OUT_DIR  / 'xgboost_scIII.json'
FIGS_DIR     = OUT_DIR  / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# ── Flags ──────────────────────────────────────────────────────────────────────
FORCE_REBUILD = False

# ── Dataset config (paper: 3000 random + 3×50×50 grilla = 10 500 pts) ─────────
N_RANDOM  = 3_000
GRID_SIZE = 50
SEED      = 42

# ── Paralelismo CPU (SMEFT19 no usa GPU; se paraleliza por procesos) ───────────
N_WORKERS = max(1, multiprocessing.cpu_count() - 1)

# ── GPU para XGBoost ───────────────────────────────────────────────────────────
def _detect_xgb_device():
    """Devuelve 'cuda' si XGBoost puede usar GPU en esta máquina, si no 'cpu'."""
    try:
        import xgboost as xgb
        dm = xgb.DMatrix(np.zeros((10, 3)), label=np.zeros(10))
        xgb.train({'device': 'cuda', 'tree_method': 'hist'}, dm,
                  num_boost_round=1, verbose_eval=False)
        return 'cuda'
    except Exception:
        return 'cpu'

XGB_DEVICE = _detect_xgb_device()

# ── Dominio físico ─────────────────────────────────────────────────────────────
C1_RANGE  = (-0.30,  0.00)
C3_RANGE  = (-0.30,  0.00)
BQ_RANGE  = ( 0.00,  3.20)
LH_MIN    = -50.0
PENALTY_K = 50.0

print(f'Dataset: {DATASET_FILE}')
print(f'Puntos : {N_RANDOM} random + 3×{GRID_SIZE}²={3*GRID_SIZE**2} grilla = {N_RANDOM+3*GRID_SIZE**2} total')
print(f'Workers: {N_WORKERS} CPUs  |  XGBoost device: {XGB_DEVICE}')

## 1. Dataset — generación o carga

El dataset `(C1, C3, bq, likelihood)` combina dos componentes, igual que en el artículo:

- **Aleatorios**: `N_RANDOM` puntos LHS en todo el hipercubo `[C1, C3, bq]`
- **Grillas best-fit**: tres grillas `GRID_SIZE × GRID_SIZE` en los planos 2D `(C1,C3)`, `(C1,bq)`, `(C3,bq)`, con el tercer parámetro fijo en el mejor ajuste

El best-fit se calcula con `SMEFT19.ellipse.minimum` y se cachea en `rotBIII.yaml`.

In [ ]:
# ── Worker globals — se inicializan una vez por proceso hijo ──────────────────
_worker_lh_fn    = None
_worker_scenario = None

def _worker_init():
    """Inicializa SMEFT19 una sola vez en cada proceso worker."""
    global _worker_lh_fn, _worker_scenario
    import SMEFT19, warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        SMEFT19.SMEFTglob.gl.make_measurement()
    _worker_lh_fn    = SMEFT19.likelihood_global
    _worker_scenario = SMEFT19.scenarios.rotBIII


def _eval_point(row):
    """Evalúa la likelihood en un punto; se ejecuta en un proceso worker."""
    C1, C3, bq = row
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            return max(float(_worker_lh_fn([C1, C3, bq], _worker_scenario)), -200.0)
        except Exception:
            return float('-inf')


# ── Helpers principales ───────────────────────────────────────────────────────

def _init_smeft():
    import SMEFT19
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        SMEFT19.SMEFTglob.gl.make_measurement()
    return SMEFT19


def _find_bestfit(smeft):
    if BESTFIT_FILE.exists():
        d  = smeft.ellipse.load(str(BESTFIT_FILE))
        bf = np.array(d['bf'])
        print(f'Best-fit cargado: C1={bf[0]:.4f}  C3={bf[1]:.4f}  bq={bf[2]:.4f}')
        return bf
    print('Calculando best-fit (puede tardar unos minutos)...')
    def neg_lh(x):
        try:    return -smeft.likelihood_global(x, smeft.scenarios.rotBIII)
        except: return -2000.0
    bf, v, d, L = smeft.ellipse.minimum(neg_lh, [-0.13, -0.13, -0.76])
    smeft.ellipse.save(bf, v, d, L, str(BESTFIT_FILE),
                       name='Mass Rotation fit, Scenario III', fit='rotBIII')
    bf = np.array(bf)
    print(f'Best-fit: C1={bf[0]:.4f}  C3={bf[1]:.4f}  bq={bf[2]:.4f}')
    return bf


def _make_grid_points(bf):
    ranges = [C1_RANGE, C3_RANGE, BQ_RANGE]
    margin = 0.02
    pts = []
    for i, j in [(0, 1), (0, 2), (1, 2)]:
        k = 3 - i - j
        lo_i, hi_i = ranges[i]; mi = margin * (hi_i - lo_i)
        lo_j, hi_j = ranges[j]; mj = margin * (hi_j - lo_j)
        xs = np.linspace(lo_i - mi, hi_i + mi, GRID_SIZE)
        ys = np.linspace(lo_j - mj, hi_j + mj, GRID_SIZE)
        for x in xs:
            for y in ys:
                row = [0.0, 0.0, 0.0]
                row[i] = x; row[j] = y; row[k] = float(bf[k])
                pts.append(row)
    return np.array(pts)


def _build_dataset(out_path):
    smeft = _init_smeft()
    bf    = _find_bestfit(smeft)

    lo = np.array([-0.30, -0.30, 0.00])
    hi = np.array([ 0.00,  0.00, 3.20])
    pts_random = scale(LatinHypercube(d=3, seed=SEED).random(n=N_RANDOM), lo, hi)
    pts_grid   = _make_grid_points(bf)
    all_pts    = np.vstack([pts_random, pts_grid])
    total      = len(all_pts)

    start     = sum(1 for _ in open(out_path)) if Path(out_path).exists() else 0
    remaining = all_pts[start:]
    print(f'\nTotal: {total} pts | pendientes: {len(remaining)} | '
          f'workers: {N_WORKERS} (SMEFT19 es CPU-only)')

    t0 = time.time()
    with multiprocessing.Pool(N_WORKERS, initializer=_worker_init) as pool, \
         open(out_path, 'at', newline='') as f:
        writer = csv.writer(f)
        for done, lg in enumerate(
            pool.imap(_eval_point, remaining.tolist(), chunksize=20), start=1
        ):
            writer.writerow([*remaining[done - 1], lg])
            f.flush()
            i = start + done
            if i % 500 == 0 or i == total:
                elapsed = time.time() - t0
                eta     = elapsed / done * (total - i) if done else 0
                print(f'  {i:6d}/{total} | {elapsed:.0f}s | ETA {eta:.0f}s', flush=True)

    print(f'Listo en {time.time()-t0:.0f}s → {out_path}')
    return pd.read_csv(out_path, names=['C1', 'C3', 'bq', 'likelihood'])


def load_or_build(path, builder, force=False):
    p = Path(path)
    if p.exists() and not force:
        print(f'Loading cached dataset: {p.name}')
        return pd.read_csv(p, names=['C1', 'C3', 'bq', 'likelihood'])
    print(f'Building dataset → {p.name}')
    return builder()


df_raw = load_or_build(
    DATASET_FILE,
    lambda: _build_dataset(DATASET_FILE),
    force=FORCE_REBUILD,
)
n_valid = (df_raw['likelihood'] > LH_MIN).sum()
print(f'Puntos totales: {len(df_raw)}  |  válidos (lh > {LH_MIN}): {n_valid}')
df_raw.describe()

## 2. Preprocesamiento

Filtrado, EDA, SMOTE **sólo en el conjunto de train** para evitar data leakage,
y KNN para asignar la likelihood a los puntos sintéticos generados por SMOTE.

In [ ]:
df = df_raw[df_raw['likelihood'] > LH_MIN].reset_index(drop=True).copy()

# EDA
pairs   = [('C1', 'C3'), ('C1', 'bq'), ('C3', 'bq')]
xlabels = [r'$C_1$', r'$C_1$', r'$C_3$']
ylabels = [r'$C_3$', r'$\beta^q$', r'$\beta^q$']
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (xc, yc), xl, yl in zip(axes, pairs, xlabels, ylabels):
    sc = ax.scatter(df[xc], df[yc], c=df['likelihood'], cmap='viridis', s=3, rasterized=True)
    plt.colorbar(sc, ax=ax, label=r'$\log\mathcal{L}$')
    ax.set_xlabel(xl); ax.set_ylabel(yl)
fig.tight_layout()
plt.savefig(FIGS_DIR / 'eda_scatter.pdf', bbox_inches='tight')
plt.show()

# Grupos para SMOTE (clase 1 = likelihood intermedia, minoría)
df['group'] = df['likelihood'].apply(lambda v: 0 if v < 16 else (1 if v < 18 else 2))
print('Grupos (0=bajo, 1=medio, 2=alto):', df['group'].value_counts().sort_index().to_dict())

X, y, g = df[['C1', 'C3', 'bq']], df['likelihood'], df['group']

# Split 84/16 como en el artículo
Xtr, Xva, ytr, yva, gtr, _ = train_test_split(
    X, y, g, test_size=0.16, random_state=SEED, stratify=g
)

Xtr_res, gtr_res = SMOTE(random_state=SEED).fit_resample(Xtr, gtr)

knn = KNeighborsRegressor(n_neighbors=5, weights='distance')
knn.fit(Xtr, ytr)
ytr_res = knn.predict(Xtr_res)

print(f'Train: {len(Xtr)} -> {len(Xtr_res)} (SMOTE)  |  Validation: {len(Xva)}')

## 3. Entrenamiento XGBoost

In [ ]:
def _plot_learning_curve(info, title='Learning curve', show=True, save_path=None):
    er     = info['eval_results']
    metric = info['eval_metric']
    keys   = list(er.keys())
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(er[keys[0]][metric], label=f'Train {metric.upper()}', lw=1.4)
    ax.plot(er[keys[1]][metric], label=f'Validation {metric.upper()}', lw=1.4)
    if info.get('best_iteration') is not None:
        ax.axvline(info['best_iteration'], color='gray', ls='--', alpha=0.6,
                   label=f"best_iter={info['best_iteration']}")
    ax.set_yscale('log'); ax.set_xlabel('Boosting iteration'); ax.set_ylabel(metric.upper())
    ax.set_title(title); ax.grid(True, which='both', ls=':', alpha=0.5); ax.legend()
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f'  saved: {save_path}')
    plt.show() if show else plt.close(fig)


es    = xgb_cb.EarlyStopping(rounds=50, data_name='validation_1', save_best=True)
model = XGBRegressor(
    n_estimators=3000, learning_rate=0.03, max_depth=6,
    tree_method='hist', device=XGB_DEVICE, eval_metric='rmse',
    random_state=SEED, callbacks=[es],
)
model.fit(
    Xtr_res, ytr_res,
    eval_set=[(Xtr_res, ytr_res), (Xva, yva)],
    verbose=False,
)

r2   = float(model.score(Xva, yva))
best = int(getattr(model, 'best_iteration', model.n_estimators - 1))
info = dict(
    eval_results   = model.evals_result(),
    eval_metric    = 'rmse',
    best_iteration = best,
    r2             = r2,
    X_val          = Xva,
)
print(f'R2 = {r2:.5f}  |  best_iter = {best}  |  device = {XGB_DEVICE}')

_plot_learning_curve(
    info,
    title=f'rotBIII · curva de aprendizaje  (R2={r2:.5f})',
    save_path=FIGS_DIR / 'learning_curve.pdf',
)

## 4. Análisis SHAP

Tres gráficas, idénticas a las de CosmoML:
1. **beeswarm + bar**: importancia global de cada feature
2. **waterfall**: descomposición aditiva para una muestra individual
3. **dependence**: efecto de cada feature coloreado por interacciones (`color=shap_values`)

In [ ]:
# ── Funciones SHAP (replicando cosmoml/ml/shap_utils.py) ──────────────────────

def _explain(model, X, n_sample=1000, seed=42):
    """TreeExplainer sobre el modelo interno; check_additivity=False como en CosmoML."""
    explainer = shap.TreeExplainer(model)
    X_s = X.sample(n=min(n_sample, len(X)), random_state=seed)
    return explainer(X_s, check_additivity=False), X_s


def shap_summary(model, X, title='', save_dir=None, n_sample=1000, show=True):
    """Beeswarm + bar (identico a CosmoML shap_summary)."""
    shap_v, X_s = _explain(model, X, n_sample=n_sample)
    for kind, fn in [('beeswarm', shap.plots.beeswarm), ('bar', shap.plots.bar)]:
        plt.figure()
        if title:
            plt.title(f'{title} ({kind})')
        fn(shap_v, show=False)
        plt.tight_layout()
        if save_dir is not None:
            p = Path(save_dir) / f'shap_{kind}.pdf'
            plt.savefig(p, dpi=300, bbox_inches='tight'); print(f'  saved: {p}')
        plt.show() if show else plt.close()
    return shap_v, X_s


def shap_waterfall(shap_values, idx=0, title='', save_path=None, show=True):
    """Waterfall para un sample (identico a CosmoML shap_waterfall)."""
    plt.figure()
    if title:
        plt.title(title)
    shap.plots.waterfall(shap_values[idx], show=False)
    plt.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight'); print(f'  saved: {save_path}')
    plt.show() if show else plt.close()


def shap_dependence_all(shap_values, X, save_dir=None, prefix='', show=True):
    """Scatter/dependence por feature, coloreado por interacciones (identico a CosmoML)."""
    for col in X.columns:
        plt.figure()
        shap.plots.scatter(shap_values[:, col], color=shap_values, show=False)
        plt.tight_layout()
        if save_dir is not None:
            p = Path(save_dir) / f'{prefix}_shap_{col}.pdf'
            plt.savefig(p, dpi=300, bbox_inches='tight'); print(f'  saved: {p}')
        plt.show() if show else plt.close()

In [ ]:
shap_v, X_s = shap_summary(
    model, info['X_val'],
    title='rotBIII · SMEFT19 likelihood',
    save_dir=FIGS_DIR,
    show=True,
)
shap_waterfall(
    shap_v, idx=0,
    title='rotBIII · waterfall (sample 0)',
    save_path=FIGS_DIR / 'shap_waterfall.pdf',
    show=True,
)
shap_dependence_all(
    shap_v, X_s,
    save_dir=FIGS_DIR,
    prefix='rotBIII',
    show=True,
)

## 5. Función de predicción con penalización de frontera

Penalizaciones exponenciales fuera del dominio de entrenamiento para evitar
extrapolaciones no físicas.

In [ ]:
def _penalty(v, lo, hi, k=PENALTY_K):
    if v < lo:   return np.exp((lo - v) * k) - 1.0
    elif v > hi: return np.exp((v - hi) * k) - 1.0
    return 0.0


def predict_point(C1, C3, bq):
    """Predice log-likelihood con penalizacion fuera del dominio fisico."""
    x   = pd.DataFrame([{'C1': C1, 'C3': C3, 'bq': bq}])
    raw = float(model.predict(x)[0])
    pen = _penalty(C1, *C1_RANGE) + _penalty(C3, *C3_RANGE) + _penalty(bq, *BQ_RANGE)
    return raw - pen


# Demo
test_pts = [(-0.10, -0.10, 1.5), (-0.25, -0.20, 2.0), (0.50, 0.0, 1.5)]
for pt in test_pts:
    print(f'predict_point{pt} = {predict_point(*pt):.4f}')

model.save_model(MODEL_OUT)
print(f'\nModelo guardado en: {MODEL_OUT}')